### Apple Watch Health Analytics
The purpose of this study is to use the following two datasets to generate insights on a simulated experimental study. Participants were asked to wear an Apple Watch and record several aspects of their activity and health at three different timepoints. Some aspects of the participants changed over the course of the trial, e.g. smoking status. These aspects were recorded in the first dataset, **health_irreg_rhythm**. 

Participants were also organized into cohorts: groups of individuals to help facilitate the study. Each cohort’s average temperature was aggregated for each month of the study and is stored in a second dataset, **health_cohort_monthly**.

The detailed schemas of the two tables are as follows:

**health_irreg_rhythm**
- id: A unique identifier for the participant
- date: Date participant recorded their data
- smoker: "yes" if the person smokes, "no" if they don't
- rings_closed: Median number of rings closed per day
- watch_type: The material type for the Apple Watch
- diff_exer_types: Median of different types of exercise types per week
- max_hr: Maximum heart rate recorded during the week
- irreg_rhythm: "1" if the person exhibited irregular heart rhythm; "0" if they did not.
- cohort: The group of the participant in the study

**health_cohort_monthly**
- cohort: The group of the participant in the study
- month: Month of the study
- mean_temp: The average temperature of the cohort during the specified month




In [0]:
-- Set current and check catalog and schema
USE CATALOG workspace;
USE SCHEMA linkprojects_apple_watch;

SELECT current_catalog(), current_schema();

In [0]:
-- Explore the schema
DESCRIBE SCHEMA EXTENDED linkprojects_apple_watch;

In [0]:
-- Check for any existing volumes.
SHOW VOLUMES

In [0]:
-- List the files inside the volume.
LIST '/Volumes/workspace/linkprojects_apple_watch/rawdata'

In [0]:
-- Preview the CSV data without creating a table. 
SELECT * 
FROM read_files('/Volumes/workspace/linkprojects_apple_watch/rawdata/health_irreg_rhythm.csv')
LIMIT 10;

Delta tables don't allow spaces in column names by default. The CSV file has column names like `Watch Type`, `Diff Exer Types`, etc., which contain spaces.

We will have to normalize the column names when creating the table — this will make downstream queries easier to write.

In [0]:
-- Create a bronze Delta table from the CSV
CREATE TABLE IF NOT EXISTS health_irreg_rhythm AS
SELECT 
  Id,
  Date,
  Smoker,
  `Rings Closed` AS Rings_Closed,
  `Watch Type` AS Watch_Type,
  `Diff Exer Types` AS Diff_Exer_Types,
  `Heart Rate` AS max_hr,
  `Irreg Rhythm` AS Irreg_Rhythm,
  Cohort
FROM read_files('/Volumes/workspace/linkprojects_apple_watch/rawdata/health_irreg_rhythm.csv',
  format => 'csv',
  header => true,
  inferSchema => true
);

In [0]:
-- Verify the bronze table created
SELECT *
FROM health_irreg_rhythm;

In the first glimpse of the bronze table, we can identify some issues with the data in the columns `Date`, `Smoker`, `max_hr`, and `Rings_Closed`. Let's fix them and create a silver table.

1. The column `Date` is a string, but it is actually a date. We need to convert it to a date.
2. The column `Smoker` is a string, but it is actually a boolean. We need to convert it to a boolean.
3. The column `max_hr` has some irregular outliers. Max heart rate 1600 is not physiologically possible. We need to replace it with the median of the column.
4. The column `Rings_Closed` has some invalid inputs. Rings closed 4 is highly likely a typo since Apple Watch only has 3 rings to close. We need to replace it with the number 3.

In [0]:
-- Create a silver Delta table from the CSV
SELECT *,
  CASE
    WHEN Date RLIKE '^[A-Za-z]{3}-[0-9]{1,2}-[0-9]{4}$' THEN to_date(Date, 'MMM-d-yyyy')
    WHEN Date RLIKE '^[0-9]{1,2}/[0-9]{1,2}/[0-9]{2}$' THEN to_date(Date, 'M/d/yy')
    ELSE NULL
  END AS Date_fixed,
  CASE
    WHEN Smoker = 'yes' THEN True
    WHEN Smoker IN ('no', 'n') THEN False
    ELSE NULL
  END AS Smoker_fixed,
  CASE
    WHEN `Heart Rate` = 1600 THEN (SELECT MEDIAN(`Heart Rate`) FROM workspace.default.health_irreg_rhythm WHERE `Heart Rate` != 1600)
    ELSE `Heart Rate`
  END AS Heart_Rate_fixed,
  CASE
    WHEN `Rings Closed` = 4 THEN 3
    ELSE `Rings Closed`
  END AS Rings_Closed_fixed
FROM workspace.linkprojects_apple_watch.health_irreg_rhythm

In [0]:
SELECT Id, Cohort, COUNT(Id)
FROM health_irreg_rhythm
GROUP BY Cohort, Id
HAVING COUNT(Id) != 3
Order BY Cohort, Id;


In [0]:
SELECT Cohort, COUNT(Id)
FROM health_irreg_rhythm
GROUP BY Cohort;

In [0]:
CREATE OR REPLACE VIEW health_irreg_rhythm_fixed AS
(
SELECT *,
  CASE
    WHEN Date RLIKE '^[A-Za-z]{3}-[0-9]{1,2}-[0-9]{4}$' THEN to_date(Date, 'MMM-d-yyyy')
    WHEN Date RLIKE '^[0-9]{1,2}/[0-9]{1,2}/[0-9]{2}$' THEN to_date(Date, 'M/d/yy')
    ELSE NULL
  END AS Date_fixed,
  CASE
    WHEN Smoker = 'yes' THEN True
    WHEN Smoker IN ('no', 'n') THEN False
    ELSE NULL
  END AS Smoker_fixed,
  CASE
    WHEN `Heart Rate` = 1600 THEN (SELECT MEDIAN(`Heart Rate`) FROM workspace.default.health_irreg_rhythm WHERE `Heart Rate` != 1600)
    ELSE `Heart Rate`
  END AS Heart_Rate_fixed,
  CASE
    WHEN `Rings Closed` = 4 THEN 3
    ELSE `Rings Closed`
  END AS Rings_Closed_fixed,
  CASE
    WHEN Cohort = 3 AND Id = 3 THEN 4
    ELSE Cohort
  END AS Cohort_fixed
FROM workspace.default.health_irreg_rhythm
)

In [0]:
WITH smoking_timeline AS (
    SELECT 
        Id,
        Date_fixed,
        Smoker_fixed,
        ROW_NUMBER() OVER (PARTITION BY Id ORDER BY Date_fixed) as timepoint,
        COUNT(*) OVER (PARTITION BY Id) as total_timepoints
    FROM health_irreg_rhythm_fixed
),
smoking_summary AS (
    SELECT 
        Id,
        total_timepoints,
        MAX(CASE WHEN timepoint = 1 THEN smoker_fixed END) as first_status,
        MAX(CASE WHEN timepoint = 2 THEN smoker_fixed END) as middle_status,
        MAX(CASE WHEN timepoint = total_timepoints THEN smoker_fixed END) as last_status,
        SUM(CASE WHEN smoker_fixed = 'yes' THEN 1 ELSE 0 END) as times_smoked
    FROM smoking_timeline
    GROUP BY id, total_timepoints
),
categorized AS (
    SELECT 
        CASE 
            -- 1. Did not change
            WHEN times_smoked = 0 OR times_smoked = total_timepoints 
                THEN 'Did Not Change'
            -- 2. Started smoking
            WHEN first_status = 'no' AND last_status = 'yes'
                THEN 'Started Smoking'
            -- 3. Quit smoking
            WHEN first_status = 'yes' AND last_status = 'no'
                THEN 'Quit Smoking'
            -- 4. Started then quit
            WHEN first_status = 'no' AND middle_status = 'yes' AND last_status = 'no'
                THEN 'Started Then Quit'
            -- 5. Quit then restarted
            WHEN first_status = 'yes' AND middle_status = 'no' AND last_status = 'yes'
                THEN 'Quit Then Restarted'
            ELSE 'Other'
        END as smoking_category
    FROM smoking_summary
)
SELECT 
    smoking_category,
    COUNT(*) as participant_count,
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(DISTINCT id) FROM health_irreg_rhythm), 1) 
        as percent_of_participants
FROM categorized
GROUP BY smoking_category;